# PH-SHOWOA GPU benchmark: RCdp1004 (pop_size=30, max_iter=1000, runs=30)


In [ ]:
INSTANCE = 'RCdp1004'
JOB_ID = '54b4f8ebc226'
RUNS = 30
MAX_ITER = 1000
POP_SIZE = 30
SEED0 = 42
GIT_URL = 'https://github.com/Welkie/ph-showoa.git'
GIT_COMMIT = 'f33b63fa7da189ec91109813e896985261226889'

In [ ]:
import os, subprocess, sys
REPO = "/tmp/ph-showoa"
if not os.path.isdir(os.path.join(REPO, "src_python_gpu_SA_RCRS_GRASP")):
    subprocess.run(["git", "init", "-q", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "add", "origin", GIT_URL], check=True)
    if subprocess.run(["git", "-C", REPO, "fetch", "-q", "--depth", "1", "origin", GIT_COMMIT]).returncode != 0:
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", "-q", GIT_COMMIT], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "checkout", "-q", "FETCH_HEAD"], check=True)
head = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code commit:", head)
assert head == GIT_COMMIT, "unexpected commit checked out"
subprocess.run(["nvidia-smi"])
from numba import cuda
import torch
if not (cuda.is_available() and torch.cuda.is_available()):
    raise RuntimeError("NO_GPU: CUDA GPU is not available in this session (GPU quota exhausted or accelerator disabled)")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import contextlib, csv, glob, io, json, os, re, sys, time, traceback
os.chdir(REPO)
sys.path.insert(0, REPO)
OUT = "/kaggle/working"
LOGS = os.path.join(OUT, "logs")
os.makedirs(LOGS, exist_ok=True)

def find_problem(name):
    fn = f"explicit_{name.lower()}.vrpsdptw"
    hits = glob.glob(f"/kaggle/input/**/Wang_Chen/{fn}", recursive=True) or glob.glob(f"/kaggle/input/**/{fn}", recursive=True)
    if not hits:
        local = os.path.join(REPO, "dataset", fn)
        hits = [local] if os.path.isfile(local) else []
    if not hits:
        raise FileNotFoundError(fn)
    return hits[0]

import src_python_gpu_SA_RCRS_GRASP.gpu_kernels as gk
import src_python_gpu_SA_RCRS_GRASP.gpu_engine as ge
_orig_bundle, _bundle_cache = gk.build_kernel_bundle, {}
def _cached_bundle(is_cuda=False):
    key = bool(is_cuda)
    if key not in _bundle_cache:
        _bundle_cache[key] = _orig_bundle(is_cuda)
    return _bundle_cache[key]
ge.build_kernel_bundle = _cached_bundle      # compile the CUDA kernels once, reuse for all runs
from src_python_gpu_SA_RCRS_GRASP.main import main as solver_main

PROBLEM = find_problem(INSTANCE)
print("Problem file:", PROBLEM)

def solve(seed, max_iter, tag):
    argv = ["ph", "--problem", PROBLEM, "--compute_backend", "cuda", "--init", "sa_rcrs_grasp",
            "--paper_flags", "--architecture", "python_cuda", "--runs", "1", "--max_iter", str(max_iter),
            "--pop_size", str(POP_SIZE), "--workers", "1", "--random_seed", str(seed)]
    buf = io.StringIO()
    t0 = time.perf_counter()
    with contextlib.redirect_stdout(buf):
        solver_main(argv)
    dt = time.perf_counter() - t0
    out = buf.getvalue()
    with open(os.path.join(LOGS, f"{tag}.log"), "w", encoding="utf-8") as f:
        f.write(out)
    m = re.search(r"Run 1 finishes \| Run Best: NV=(\d+), TD=([\d.]+)", out)
    if not m:
        raise RuntimeError(f"could not parse result of {tag}:\n" + out[-2000:])
    return int(m.group(1)), float(m.group(2)), dt

t0 = time.perf_counter()
solve(SEED0, 60, "warmup")                    # JIT-compiles every kernel (LS@25, migration@20, ...), not timed
print(f"Warm-up (JIT compile) took {time.perf_counter() - t0:.1f}s", flush=True)

rows = []
runs_path = os.path.join(OUT, "runs.csv")
for r in range(1, RUNS + 1):
    seed = SEED0 + r - 1
    nv, td, dt = solve(seed, MAX_ITER, f"run{r:02d}_seed{seed}")
    rows.append({"instance": INSTANCE, "run": r, "seed": seed, "NV": nv, "TD": round(td, 4),
                 "cost": round(2000 * nv + td, 4), "time": round(dt, 3)})
    with open(runs_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)
    print(f"[run {r:2d}/{RUNS}] seed={seed} NV={nv} TD={td:.4f} time={dt:.1f}s", flush=True)

best = min(rows, key=lambda x: (x["NV"], x["TD"]))
summary = {"job_id": JOB_ID, "instance": INSTANCE, "status": "Success", "runs": len(rows),
           "best_NV": best["NV"], "best_TD": best["TD"],
           "avg_time_s": round(sum(x["time"] for x in rows) / len(rows), 3), "commit": GIT_COMMIT}
with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)
print("SUMMARY:", json.dumps(summary))